# Обработка больших данных с Apache Spark Часть 2

## Практика

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Создаем Spark сессию
spark = SparkSession.builder \
    .appName("Calls Analysis") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [2]:
spark

In [3]:
sc = spark.sparkContext

In [4]:
df = spark.read.csv('calls_sample_data_homework.csv', header=True)#.repartition(8, "region_code")

In [8]:
# Исправление типов данных
df = (df.withColumn("monthly_fee",col("monthly_fee").cast("float"))
      .withColumn("call_start",col("call_start").cast("timestamp"))
      .withColumn("call_end",col("call_end").cast("timestamp"))
)

In [9]:
df.printSchema()

root
 |-- call_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- call_start: timestamp (nullable = true)
 |-- call_end: timestamp (nullable = true)
 |-- region_code: string (nullable = true)
 |-- tariff_name: string (nullable = true)
 |-- monthly_fee: float (nullable = true)



### Shuffle

In [10]:


# 1. ПРИМЕР С RDD (как в оригинальном примере)
print("\n=== ПРИМЕР С RDD ===")
# Преобразуем DataFrame в RDD пар (тариф, 1) для подсчета
df.rdd.getNumPartitions()
print(f"Изначально партиций в RDD: {df.rdd.getNumPartitions()}")


print("\n=== ГРУППИРОВКА ПО РЕГИОНУ ===")
calls_by_region = df.groupBy("region_code")

print("План выполнения группировки по региону:")
calls_by_region.count().explain()

result_region = calls_by_region.count().orderBy("count", ascending=False).limit(5).collect()
print(f"\nТоп-5 регионов по количеству звонков: {result_region}")


=== ПРИМЕР С RDD ===
Изначально партиций в RDD: 1

=== ГРУППИРОВКА ПО РЕГИОНУ ===
План выполнения группировки по региону:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region_code#21], functions=[count(1)])
   +- Exchange hashpartitioning(region_code#21, 200), ENSURE_REQUIREMENTS, [plan_id=41]
      +- HashAggregate(keys=[region_code#21], functions=[partial_count(1)])
         +- FileScan csv [region_code#21] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/calls_sample_data_homework.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<region_code:string>



Топ-5 регионов по количеству звонков: [Row(region_code='77', count=100), Row(region_code='99', count=85), Row(region_code='50', count=82), Row(region_code='47', count=81), Row(region_code='11', count=80)]


In [11]:


# 5. ПРИМЕР БЕЗ SHUFFLE (узкие трансформации)
print("\n=== ПРИМЕР БЕЗ SHUFFLE ===")
# Фильтрация - НЕ вызывает shuffle (узкая трансформация)
filtered_df = df.filter(col("monthly_fee") > 300)

print("План выполнения фильтрации:")
filtered_df.explain()

# Простая агрегация с фильтрацией
no_shuffle_example = filtered_df.select("tariff_name").distinct()
print(f"\nТарифы с абонплатой > 300: {no_shuffle_example.collect()}")

# 6. Визуализация разницы в планах выполнения
print("\n=== СРАВНЕНИЕ ПЛАНОВ ВЫПОЛНЕНИЯ ===")
print("1. План с shuffle (groupBy):")
df.groupBy("tariff_name").count().explain()

print("\n2. План без shuffle (filter):")
df.filter(col("region_code") == 29).explain()



=== ПРИМЕР БЕЗ SHUFFLE ===
План выполнения фильтрации:
== Physical Plan ==
*(1) Project [call_id#17, subscriber_id#18, cast(call_start#19 as timestamp) AS call_start#36, cast(call_end#20 as timestamp) AS call_end#37, region_code#21, tariff_name#22, cast(monthly_fee#23 as float) AS monthly_fee#35]
+- *(1) Filter (isnotnull(monthly_fee#23) AND (cast(monthly_fee#23 as float) > 300.0))
   +- FileScan csv [call_id#17,subscriber_id#18,call_start#19,call_end#20,region_code#21,tariff_name#22,monthly_fee#23] Batched: false, DataFilters: [isnotnull(monthly_fee#23), (cast(monthly_fee#23 as float) > 300.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/calls_sample_data_homework.csv], PartitionFilters: [], PushedFilters: [IsNotNull(monthly_fee)], ReadSchema: struct<call_id:string,subscriber_id:string,call_start:string,call_end:string,region_code:string,t...



Тарифы с абонплатой > 300: [Row(tariff_name='Премиум'), Row(tariff_name='Стандарт'), Row(tariff_name='Безлимит'), Row(t

## Broadcast

In [12]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

In [13]:
regions_data = [
    (i, f"Регион_{i}", f"Страна_{i%10}", ["Север", "Юг", "Запад", "Восток"][i % 4])
    for i in range(100)  # 100 записей - маленькая таблица
]

regions_schema = StructType([
    StructField("region_code", IntegerType(), True),
    StructField("region_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("zone", StringType(), True)
])

regions_df = spark.createDataFrame(regions_data, schema=regions_schema)

# Создаем справочник тарифов (тоже маленький)
tariffs_data = [
    ("Базовый", 300, 100, 5, "Экономный"),
    ("Стандарт", 500, 500, 10, "Базовый"),
    ("Премиум", 800, 1000, 20, "Премиум"),
    ("Бизнес", 1200, 5000, 50, "Бизнес")
]

tariffs_df = spark.createDataFrame(tariffs_data,
    ["tariff_name", "monthly_fee", "minutes_included", "sms_included", "tariff_type"])

print("Справочник регионов:")
regions_df.show(10)
print(f"Размер справочника регионов: {regions_df.count()} записей")

print("\nСправочник тарифов:")
tariffs_df.show()
print(f"Размер справочника тарифов: {tariffs_df.count()} записей")

Справочник регионов:
+-----------+-----------+--------+------+
|region_code|region_name| country|  zone|
+-----------+-----------+--------+------+
|          0|   Регион_0|Страна_0| Север|
|          1|   Регион_1|Страна_1|    Юг|
|          2|   Регион_2|Страна_2| Запад|
|          3|   Регион_3|Страна_3|Восток|
|          4|   Регион_4|Страна_4| Север|
|          5|   Регион_5|Страна_5|    Юг|
|          6|   Регион_6|Страна_6| Запад|
|          7|   Регион_7|Страна_7|Восток|
|          8|   Регион_8|Страна_8| Север|
|          9|   Регион_9|Страна_9|    Юг|
+-----------+-----------+--------+------+
only showing top 10 rows
Размер справочника регионов: 100 записей

Справочник тарифов:
+-----------+-----------+----------------+------------+-----------+
|tariff_name|monthly_fee|minutes_included|sms_included|tariff_type|
+-----------+-----------+----------------+------------+-----------+
|    Базовый|        300|             100|           5|  Экономный|
|   Стандарт|        500|       

In [14]:
print("=" * 80)
print("ПРИМЕР 1: SHUFFLE JOIN (МЕДЛЕННЫЙ СПОСОБ)")
print("=" * 80)

# Засекаем время
start_time = time.time()

# Обычный join - вызовет SHUFFLE обоих DataFrame
result_shuffle = df.join(
    regions_df,
    df.region_code == regions_df.region_code,
    "left"
)

# Выполняем действие для запуска вычислений
count_shuffle = result_shuffle.count()

end_time = time.time()
shuffle_time = end_time - start_time

print(f"\nРезультат shuffle join:")
print(f"Количество записей после join: {count_shuffle}")
print(f"Время выполнения: {shuffle_time:.2f} секунд")

# Покажем план выполнения (ищем Exchange = SHUFFLE)
print("\nПлан выполнения (Shuffle Join):")
result_shuffle.explain()

# Покажем первые 5 результатов
print("\nПервые 5 результатов (Shuffle Join):")
result_shuffle.show(5)

ПРИМЕР 1: SHUFFLE JOIN (МЕДЛЕННЫЙ СПОСОБ)

Результат shuffle join:
Количество записей после join: 1000
Время выполнения: 2.54 секунд

План выполнения (Shuffle Join):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [cast(region_code#21 as bigint)], [cast(region_code#74 as bigint)], LeftOuter
   :- Sort [cast(region_code#21 as bigint) ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(cast(region_code#21 as bigint), 200), ENSURE_REQUIREMENTS, [plan_id=473]
   :     +- Project [call_id#17, subscriber_id#18, cast(call_start#19 as timestamp) AS call_start#36, cast(call_end#20 as timestamp) AS call_end#37, region_code#21, tariff_name#22, cast(monthly_fee#23 as float) AS monthly_fee#35]
   :        +- FileScan csv [call_id#17,subscriber_id#18,call_start#19,call_end#20,region_code#21,tariff_name#22,monthly_fee#23] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/calls_sample_data_homework.csv], PartitionFilters

In [15]:
print("\n" + "=" * 80)
print("ПРИМЕР 2: BROADCAST JOIN (БЫСТРЫЙ СПОСОБ)")
print("=" * 80)

# Засекаем время
start_time = time.time()

# Broadcast join - отправляем маленький DataFrame на все узлы
result_broadcast = df.join(
    broadcast(regions_df),  # Ключевое слово broadcast!
    df.region_code == regions_df.region_code,
    "left"
)

# Выполняем действие для запуска вычислений
count_broadcast = result_broadcast.count()

end_time = time.time()
broadcast_time = end_time - start_time

print(f"\nРезультат broadcast join:")
print(f"Количество записей после join: {count_broadcast}")
print(f"Время выполнения: {broadcast_time:.2f} секунд")

# Покажем план выполнения (ищем BroadcastExchange)
print("\nПлан выполнения (Broadcast Join):")
result_broadcast.explain()

# Покажем первые 5 результатов
print("\nПервые 5 результатов (Broadcast Join):")
result_broadcast.show(5)


ПРИМЕР 2: BROADCAST JOIN (БЫСТРЫЙ СПОСОБ)

Результат broadcast join:
Количество записей после join: 1000
Время выполнения: 0.77 секунд

План выполнения (Broadcast Join):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [cast(region_code#21 as bigint)], [cast(region_code#74 as bigint)], LeftOuter, BuildRight, false
   :- Project [call_id#17, subscriber_id#18, cast(call_start#19 as timestamp) AS call_start#36, cast(call_end#20 as timestamp) AS call_end#37, region_code#21, tariff_name#22, cast(monthly_fee#23 as float) AS monthly_fee#35]
   :  +- FileScan csv [call_id#17,subscriber_id#18,call_start#19,call_end#20,region_code#21,tariff_name#22,monthly_fee#23] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/calls_sample_data_homework.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<call_id:string,subscriber_id:string,call_start:string,call_end:string,region_code:string,t...
   +- BroadcastExchange 

# Домашнее задание

Выполните задачи на pyspark.
В каждом решении дополнительно выполните explain и изучите, какой вид трансформации выполняется (узкая, широкая)

## Задание 1: Вывести список call_id и длительность в минутах

## Задание 2: Вывести список самых длинных call_id и длительность в минутах в порядке убывания длительности

## Задание 3: Вывести все звонки по тарифу Безлимит

## Задание 4: Вывести все звонки по тарифу Безлимит в порядке возрастания длительности

## Задание 5: Составьте список уникальных абонентов с указанием их региона, тарифа, абонентской платы, количества совершенных звонков, суммарной длительности звонков

## Задание 6: По списку из предыдущего задания посчитайте размер выручки по всем абонентам за один месяц (сумма всех оплат)

## Задание 7: По списку из предыдущего задания посчитайте размер плановой выручки по всем абонентам в год

## Задание 8: По списку из предыдущего задания посчитайте размер выручки по тарифам

## Задание 9: Найдите суммарную длительность всех звонков



##Задание 10: Найдите среднюю выручку на минуту разговора как отношение суммарной выручки к суммарной длительности в минутах

In [ ]:

# Остановка Spark сессии
spark.stop()